# 01. Introduction to LangGraph — state based agent orchestration framework

## Learning Objectives

Understand the core concepts of LangGraph and two APIs (Graph API, Functional API).

## 1.1 What is LangGraph?

LangGraph is a **low-level orchestration framework** for the LangChain ecosystem.

### LangChain 3-tier structure

| tier | Role | Description |
|------|------|------|
| Deep Agents | high standard | Pre-built agent system |
| LangChain | agent | Building LLM agent tool |
| **LangGraph** | **workflow** | **state-based orchestration** |

### Key Features

- **state Management**: TypedDict-based state definition and reducer
- **Persistence**: Auto-save state via checkpointer
- **streaming**: Real-time token unit output
- **Human-in-the-loop**: interrupt/resume for human intervention
- **durable execution**: Automatic recovery in case of failure

## 1.2 Core concepts

LangGraph defines workflow based on **graph structure**.

### Component

| concept | Description |
|------|------|
| **Node** | Processing unit — defined as a Python function |
| **Edge** | Connection between nodes, conditional branching possible |
| **state(State)** | Defined as TypedDict, shared data between nodes |
| **checkpointer(Checkpointer)** | Auto-save each step state |

### Graph structure diagram

![StateGraph 구조 — START→NodeA→조건분기→END](../assets/images/stategraph_structure.png)

## 1.3 Two APIs

LangGraph provides an API that allows the same functionality to be implemented in two different styles.

| Characteristics | Graph API | Functional API |
|------|----------|----------------|
| approach | declarative (node+edge) | imperative (Python control flow) |
| state Management | Explicit State + Reducer | No need for function scope or reducer |
| Visualization | Graph visualization support | Not supported |
| checkpointing | New checkpoint every superstep | By `@task`, save to existing checkpoint |
| suitable situation | Complex workflow, Team Development | Migrate existing code, simple flow |

## 1.4 Environment Setup and check installation

Verify that the required packages are installed correctly.

In [ ]:
import importlib

packages = {
    "langgraph": "langgraph",
    "langchain": "langchain",
    "langchain_openai": "langchain-openai",
}

print("=" * 50)
print("LangGraph Check environment")
print("=" * 50)

for module_name, package_name in packages.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "installed")
        print(f"  OK  {package_name}: {version}")
    except ImportError:
        print(f"  ERR {package_name}: 설치되지 않음")

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv(override=True)

required = ["OPENAI_API_KEY"]
optional = ["TAVILY_API_KEY", "LANGSMITH_API_KEY"]

print("API key state:")
for key in required:
    print(f"  {'OK' if os.environ.get(key) else 'MISSING'} {key} (필수)")
for key in optional:
    print(f"  {'OK' if os.environ.get(key) else '--'} {key} (선택)")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically activated when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


In [ ]:
# Core import verification
from langgraph.graph import StateGraph, START, END
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt
from langchain.tools import tool
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI

print("All core imports completed")

## 1.5 A taste of the Graph API

The Graph API defines workflow in a **declarative** manner.

1. Create a graph builder with `StateGraph(State)` — `state_schema`
2. `add_node()` — Node (function) registration
3. `add_edge()` — Connections between nodes
4. `compile()` — Create an executable graph
5. `invoke()` — Graph execution

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class SimpleState(TypedDict):
    input: str
    output: str

def greet(state: SimpleState) -> dict:
    return {"output": f"안녕하세요, {state['input']}님!"}

builder = StateGraph(SimpleState)
builder.add_node("greet", greet)
builder.add_edge(START, "greet")
builder.add_edge("greet", END)

graph = builder.compile()
result = graph.invoke({"input": "LangGraph"}, config=lf_config)
print("Graph API results:", result["output"])

## 1.6 A taste of the Functional API

The Functional API defines workflow in an **imperative** manner.

- `@task` — Unit operation definition (checkpointing unit)
- `@entrypoint` — workflow Entry point definition
- Use regular Python control flow (`if`, `for`, `while`, etc.)

In [ ]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

@task
def process(text: str) -> str:
    return f"처리 완료: {text.upper()}"

@entrypoint(checkpointer=InMemorySaver())
def simple_workflow(text: str) -> str:
    result = process(text).result()
    return result

output = simple_workflow.invoke("hello langgraph", {**{"configurable": {"thread_id": "demo"}}, **lf_config})
print("Functional API results:", output)

## 1.7 Next Steps

In the next Note book, we will learn Graph API in earnest.

- **02_graph_api.ipynb** — StateGraph, node, edge, conditional branch, state reducer

## Summary

| Item | Description |
|------|------|
| LangGraph | state Based on agent Orchestration Framework |
| Graph API | Explicit state flow definition with `StateGraph` |
| Functional API | `@entrypoint` + `@task` Functional workflow |
| Key concepts | State (state), Node, Edge |
| checkpointer | state Persistence, multi-turn dialogue, time travel support |

### Next Steps
→ **[02_graph_api.ipynb](./02_graph_api.ipynb)**: Learn the core concepts of Graph API.